# Heart failure survival with MIMIC-III
### Kaplan-Meier, Cox model and Random Survival Forest

Self-learning tutorial. I wanted to learn **survival analysis**, so I tried it on heart failure patients in MIMIC-III.

**Question:** after a heart failure admission, who dies in the next 90 days, how fast, and what predicts it?

**What you will learn**
1. Pull a cohort from MIMIC-III on BigQuery with SQL
2. Turn it into survival data (a `time` and an `event` for each patient)
3. Kaplan-Meier curves and the log-rank test
4. Cox proportional hazards model (hazard ratios)
5. Random Survival Forest, a machine learning model for survival
6. Compare the models with the C-index and see what they learned

**To run it yourself**
- You need PhysioNet credentialed access to MIMIC-III v1.4, linked to BigQuery
- Open it in Colab or BigQuery Studio, change `PROJECT_ID` in the setup cell to your own Google Cloud project, then Run all (about 5 min)
- Query results are saved to `data/` so reruns dont hit BigQuery again
- I only print counts and summaries, never raw patient rows, since MIMIC data cant be shared

In [ ]:
# lifelines              -> Kaplan-Meier curves, log-rank test, Cox model
# scikit-survival        -> Random Survival Forest and the C-index
# google-cloud-bigquery  -> runs the SQL, db-dtypes lets pandas read the result
# (pip might update scikit-learn to a version scikit-survival works with, thats ok)
!pip install -q lifelines scikit-survival google-cloud-bigquery db-dtypes
print("done")

## Setup

Imports, settings and the BigQuery connection. The only thing you should need to change is `PROJECT_ID`.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from google.cloud import bigquery

import lifelines
import sksurv
import sklearn
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test, proportional_hazard_test
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

# versions I used, in case yours give slightly different numbers
print("lifelines", lifelines.__version__, "| scikit-survival", sksurv.__version__,
      "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

PROJECT_ID = "mimic-508001"   # <- change this to your own Google Cloud project id
SEED = 42                     # fixed seed so the split and the forest come out the same every run
HORIZON = 90                  # days of follow up after admission (step 3 explains why 90)

os.makedirs("data", exist_ok=True)      # query results get cached here as csv
os.makedirs("figures", exist_ok=True)   # every plot also gets saved here as a png
R = {}                                  # key numbers, printed in the summary at the end

# one plain style for every plot
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b", "axes.titlesize": 13,
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "legend.frameon": False,
})
BLUE, ORANGE, GRAY, INK = "#2a78d6", "#eb6834", "#9a9893", "#0b0b0b"
RAMP = ["#86b6ef", "#2a78d6", "#104281"]   # light to dark blue, for ordered groups (low/mid/high)

# readable names for the features in plots and tables
NAMES = {"age": "Age", "female": "Female", "emergency": "Emergency admission",
         "diabetes": "Diabetes", "ckd": "Chronic kidney disease", "afib": "Atrial fibrillation",
         "copd": "COPD", "met_cancer": "Metastatic cancer", "creatinine": "Creatinine (log)",
         "bun": "BUN (log)", "sodium": "Sodium", "hemoglobin": "Hemoglobin"}


def save(fig, name):
    """Save a figure into figures/ as a png and show it."""
    fig.savefig(f"figures/{name}.png", bbox_inches="tight", facecolor="white")
    plt.show()


def fmt_p(p):
    """p values in the usual way, tiny ones as p < 0.001."""
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

In [ ]:
# Colab needs you to log in. BigQuery Studio is already logged in
# (it might print a warning about authenticate_user here, thats fine)
IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()

client = bigquery.Client(project=PROJECT_ID)
# with a default dataset I can write "admissions" instead of "physionet-data.mimiciii_clinical.admissions"
job_config = bigquery.QueryJobConfig(default_dataset="physionet-data.mimiciii_clinical")


def run_query(sql, name):
    """Run the sql on BigQuery once and cache the result as data/<name>.csv.
    If the csv is already there it just reads it, so reruns are fast and free."""
    path = f"data/{name}.csv"
    if not os.path.exists(path):
        client.query(sql, job_config=job_config).to_dataframe().to_csv(path, index=False)
    # always read it back from the csv, so the first run and reruns get the same column types
    return pd.read_csv(path)

## Step 1: the heart failure cohort

One row per patient: their **first** admission with a heart failure code (ICD-9 starting with 428, same definition as my SQL assignment).

- `admissions` gives the admit time and admission type
- `patients` gives gender, birth date and date of death (`dod`)
- `days_to_death` comes out empty for people with no recorded death

In [ ]:
sql_cohort = """
WITH hf_adm AS (              -- every admission that has a heart failure code
  SELECT DISTINCT hadm_id
  FROM diagnoses_icd
  WHERE icd9_code LIKE '428%'
),
ranked AS (                   -- number each patient's HF admissions in time order
  SELECT
    a.subject_id, a.hadm_id, a.admittime, a.admission_type,
    p.gender, p.dob, p.dod,
    ROW_NUMBER() OVER (PARTITION BY a.subject_id ORDER BY a.admittime) AS rn
  FROM admissions a
  JOIN hf_adm h   ON h.hadm_id = a.hadm_id
  JOIN patients p ON p.subject_id = a.subject_id
)
SELECT
  subject_id, hadm_id, gender, admission_type,
  DATE_DIFF(DATE(admittime), DATE(dob), DAY) / 365.25 AS age,
  DATE_DIFF(DATE(dod), DATE(admittime), DAY)          AS days_to_death
FROM ranked
WHERE rn = 1                  -- keep only the first HF admission
"""
cohort = run_query(sql_cohort, "cohort")
R["n_cohort_raw"] = len(cohort)
print(f"{len(cohort):,} patients, each with their first heart failure admission")

## Step 2: other diagnoses and first labs

Things that could change how long someone survives.

- **Comorbidities** from `diagnoses_icd` on the same admission: diabetes, chronic kidney disease (CKD), atrial fibrillation, COPD, metastatic cancer
- **Labs** from `labevents`: the first creatinine, BUN, sodium and hemoglobin, up to 24 hours after admission. Early values, so the models only use info you'd have near the start of the stay.

In [ ]:
# ICD-9 codes I used:
#   250.xx diabetes | 585.x chronic kidney disease | 427.31 atrial fibrillation
#   491, 492, 496 COPD (chronic bronchitis, emphysema, chronic airway obstruction)
#   196 to 199 metastatic (secondary) cancer
# MAX(IF(...)) = 1 if any diagnosis on that admission matches, otherwise 0
sql_comorb = """
SELECT
  hadm_id,
  MAX(IF(icd9_code LIKE '250%', 1, 0))                             AS diabetes,
  MAX(IF(icd9_code LIKE '585%', 1, 0))                             AS ckd,
  MAX(IF(icd9_code = '42731', 1, 0))                               AS afib,
  MAX(IF(SUBSTR(icd9_code, 1, 3) IN ('491', '492', '496'), 1, 0))  AS copd,
  MAX(IF(SUBSTR(icd9_code, 1, 3) BETWEEN '196' AND '199', 1, 0))   AS met_cancer
FROM diagnoses_icd
WHERE hadm_id IN (SELECT hadm_id FROM diagnoses_icd WHERE icd9_code LIKE '428%')
GROUP BY hadm_id
"""
comorb = run_query(sql_comorb, "comorb")
print(f"comorbidities for {len(comorb):,} heart failure admissions")

In [ ]:
# itemids from d_labitems: 50912 creatinine, 51006 urea nitrogen (BUN), 50983 sodium, 51222 hemoglobin
# labs drawn in the ER right before admission count too, as long as they're linked to the admission
sql_labs = """
WITH hf_adm AS (
  SELECT DISTINCT hadm_id FROM diagnoses_icd WHERE icd9_code LIKE '428%'
),
first_labs AS (
  SELECT
    l.hadm_id, l.itemid, l.valuenum,
    ROW_NUMBER() OVER (PARTITION BY l.hadm_id, l.itemid ORDER BY l.charttime) AS rn
  FROM labevents l
  JOIN hf_adm h     ON h.hadm_id = l.hadm_id
  JOIN admissions a ON a.hadm_id = l.hadm_id
  WHERE l.itemid IN (50912, 51006, 50983, 51222)
    AND l.valuenum > 0                                          -- skips empty and zero values
    AND l.charttime <= DATETIME_ADD(a.admittime, INTERVAL 24 HOUR)
)
SELECT
  hadm_id,
  MAX(IF(itemid = 50912, valuenum, NULL)) AS creatinine,
  MAX(IF(itemid = 51006, valuenum, NULL)) AS bun,
  MAX(IF(itemid = 50983, valuenum, NULL)) AS sodium,
  MAX(IF(itemid = 51222, valuenum, NULL)) AS hemoglobin
FROM first_labs
WHERE rn = 1          -- rn = 1 is the first value of each lab
GROUP BY hadm_id      -- turns the 4 lab rows into 4 columns
"""
labs = run_query(sql_labs, "labs")
print(f"labs for {len(labs):,} heart failure admissions")

In [ ]:
# left joins so nobody gets dropped just because they had no labs
df = cohort.merge(comorb, on="hadm_id", how="left").merge(labs, on="hadm_id", how="left")

COMORB = ["diabetes", "ckd", "afib", "copd", "met_cancer"]
LABS = ["creatinine", "bun", "sodium", "hemoglobin"]
df[COMORB] = df[COMORB].fillna(0).astype(int)

R["lab_missing"] = (df[LABS].isna().mean() * 100).round(1).to_dict()
print("missing labs (% of patients):")
print((df[LABS].isna().mean() * 100).round(1).to_string())

## Step 3: turn it into survival data

Survival analysis needs 2 columns for each patient:

- `time`: days from admission until they died, or until we stopped following them
- `event`: 1 if they died in that time, 0 if they were still alive at the end (**censored**)

Censored doesnt mean they lived forever, only that we know they were alive up to that day. A normal yes/no classifier would have to call them "survived" and it ignores *when* people died. Survival methods use both.

**Why 90 days?** MIMIC gets deaths after discharge from Social Security records. For the newer MetaVision patients (2008 to 2012) those only go about 90 days out, for the older CareVue patients up to 4 years. If I followed people for a whole year I'd miss deaths in the newer group. 90 days from admission is covered for everyone.

First a made up example (not MIMIC data) to show what this looks like:

In [ ]:
# 5 made up patients: two die (X), three are still alive at day 90 (censored, open circle)
toy = pd.DataFrame({"patient": ["A", "B", "C", "D", "E"],
                    "time":    [12, 90, 55, 90, 90],
                    "event":   [1, 0, 1, 0, 0]})

fig, ax = plt.subplots(figsize=(8, 3.6))
for i, row in toy.iterrows():
    ax.plot([0, row["time"]], [i, i], color=GRAY, lw=2)
    if row["event"] == 1:
        ax.scatter(row["time"], i, marker="X", s=140, color=ORANGE, zorder=3)
        ax.annotate(f"died, time = {row['time']}, event = 1", (row["time"], i),
                    xytext=(10, 0), textcoords="offset points", va="center", fontsize=10)
    else:
        ax.scatter(row["time"], i, marker="o", s=100, facecolor="white", edgecolor=BLUE, lw=2, zorder=3)
ax.axvline(HORIZON, color="#52514e", ls="--", lw=1)
ax.text(HORIZON, len(toy) - 0.4, "end of follow up", ha="right", fontsize=10, color="#52514e")
ax.set_yticks(range(len(toy)), [f"Patient {p}" for p in toy["patient"]])
ax.set_ylim(-0.6, len(toy) - 0.2)
ax.set_xlim(0, HORIZON + 4)
ax.set_xlabel("Days since admission")
ax.grid(axis="y", visible=False)
ax.legend(handles=[
    Line2D([], [], marker="X", ls="", color=ORANGE, markersize=10, label="died (event = 1)"),
    Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor=BLUE, mew=2,
           markersize=9, label="alive at day 90, censored (event = 0)")],
    loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2)
ax.set_title("Made up example: time and event for 5 patients", loc="left")
save(fig, "fig0_censoring_example")

In [ ]:
# a death date before the admission would be a data error, drop it if there are any
bad = df["days_to_death"] < 0
print("death date before admission:", int(bad.sum()))
df = df[~bad].copy()

# event = 1 if they died within 90 days of admission
# (days_to_death is NaN when there's no death date, and NaN <= 90 is False, so those get 0)
df["event"] = (df["days_to_death"] <= HORIZON).astype(int)

# time = day of death for those who died, otherwise 90 (alive at day 90 = censored)
df["time"] = np.where(df["event"] == 1, df["days_to_death"], HORIZON).astype(float)

# dod is only a date, so dying on the admission day gives time 0. I use half a day instead
df["time"] = df["time"].clip(lower=0.5)

print(df[["time", "event"]].describe().round(2).to_string())

## Step 4: clean up the features

- MIMIC hides the real age of anyone over 89 by shifting their birth date, so their age comes out around 300. I set those to 90.
- Gender and admission type become 0/1 columns.
- Missing labs get filled in step 7, after the train/test split.

In [ ]:
df.loc[df["age"] > 89, "age"] = 90
df = df[df["age"] >= 18].copy()      # adults only (pretty much everyone with HF already is)

df["female"] = (df["gender"] == "F").astype(int)
df["emergency"] = df["admission_type"].isin(["EMERGENCY", "URGENT"]).astype(int)

FEATURES = ["age", "female", "emergency"] + COMORB + LABS

# cohort summary, only counts and percents
R["n"] = len(df)
R["deaths"] = int(df["event"].sum())
R["death_pct"] = 100 * df["event"].mean()
R["median_age"] = float(df["age"].median())
R["pct"] = {c: 100 * df[c].mean() for c in ["female", "emergency"] + COMORB}
R["median_days_to_death"] = float(df.loc[df["event"] == 1, "time"].median())

summary = pd.Series({
    "patients": f"{R['n']:,}",
    "died within 90 days": f"{R['deaths']:,} ({R['death_pct']:.1f}%)",
    "median age": f"{R['median_age']:.0f}",
    **{NAMES[c]: f"{v:.1f}%" for c, v in R["pct"].items()},
})
print(summary.to_string())

## Step 5: Kaplan-Meier curve

The Kaplan-Meier (KM) curve is the basic survival plot. Every time someone dies, the curve steps down by (deaths that day) / (people still being followed that day). Censored people count as "at risk" until they leave, then just drop out of the denominator, they never count as deaths.

The shaded band is the 95% confidence interval. The table under the plot is how many people are still being followed at each time.

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(df["time"], event_observed=df["event"], label="All heart failure patients")

R["km_30"] = 100 * float(kmf.survival_function_at_times(30).iloc[0])
R["km_90"] = 100 * float(kmf.survival_function_at_times(HORIZON).iloc[0])
print(f"alive at 30 days: {R['km_30']:.1f}%   alive at 90 days: {R['km_90']:.1f}%")
print(f"half of the 90 day deaths happened by day {R['median_days_to_death']:.0f}")


def y_floor(lowest):
    """Bottom of the y axis: a bit under the lowest curve, rounded down to 0.1."""
    return max(0.0, np.floor(lowest * 10) / 10 - 0.1)


fig, ax = plt.subplots(figsize=(8, 5.4))
kmf.plot_survival_function(ax=ax, color=BLUE, lw=2, ci_alpha=0.2)
ax.set_xlim(0, HORIZON)
ax.set_xticks(range(0, HORIZON + 1, 15))
ax.set_ylim(y_floor(R["km_90"] / 100), 1.0)
ax.set_xlabel("Days since admission")
ax.set_ylabel("Probability still alive")
ax.set_title("Survival after a heart failure admission (Kaplan-Meier)", loc="left")
ax.get_legend().remove()
# "at risk" row under the plot = people still being followed (time >= t) at each tick
ax.text(0, -0.2, "Number at risk (still being followed)", transform=ax.transAxes, fontsize=10, color="#52514e")
for t in range(0, HORIZON + 1, 15):
    ax.text(t, -0.26, f"{(df['time'] >= t).sum():,}", transform=ax.get_xaxis_transform(),
            ha="center", fontsize=10, color="#52514e")
save(fig, "fig1_km_overall")

## Step 6: compare groups with the log-rank test

Same curve, but one per group. The **log-rank test** checks if the curves are really different or if the gap could just be chance. It compares how many deaths each group had with how many you'd expect if all groups had the same survival. Small p value (< 0.05) = real difference.

I compare age groups and CKD vs no CKD.

In [ ]:
# right=False makes the bins [18, 65), [65, 80), [80, 91)
df["age_group"] = pd.cut(df["age"], bins=[18, 65, 80, 91], right=False,
                         labels=["Under 65", "65 to 79", "80 and over"]).astype(str)


def km_by_group(ax, col, levels, labels, colors, title):
    """One KM curve per group, with the day 90 survival % written at the end of each line.
    Returns the 90 day death % for each group."""
    out = {}
    for lev, lab, color in zip(levels, labels, colors):
        d = df[df[col] == lev]
        k = KaplanMeierFitter().fit(d["time"], d["event"], label=f"{lab} (n={len(d):,})")
        k.plot_survival_function(ax=ax, color=color, lw=2, ci_show=False)
        s90 = float(k.survival_function_at_times(HORIZON).iloc[0])
        ax.annotate(f"{100 * s90:.0f}%", (HORIZON, s90), xytext=(5, 0),
                    textcoords="offset points", va="center", fontsize=11, color=INK)
        out[lab] = 100 * (1 - s90)
    ax.set_xlim(0, HORIZON)
    ax.set_xticks(range(0, HORIZON + 1, 15))
    ax.set_xlabel("Days since admission")
    ax.set_title(title, loc="left")
    ax.legend(loc="lower left")
    return out


fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
age_levels = ["Under 65", "65 to 79", "80 and over"]
R["death_by_age"] = km_by_group(axes[0], "age_group", age_levels, age_levels, RAMP, "By age group")
R["death_by_ckd"] = km_by_group(axes[1], "ckd", [0, 1], ["No CKD", "CKD"], [BLUE, ORANGE],
                                "By chronic kidney disease")
axes[0].set_ylabel("Probability still alive")
lowest = 1 - max(list(R["death_by_age"].values()) + list(R["death_by_ckd"].values())) / 100
axes[0].set_ylim(y_floor(lowest), 1.0)

# log-rank tests: multivariate version for 3 groups, normal one for 2 groups
lr_age = multivariate_logrank_test(df["time"], df["age_group"], df["event"])
ckd1, ckd0 = df[df["ckd"] == 1], df[df["ckd"] == 0]
lr_ckd = logrank_test(ckd1["time"], ckd0["time"], ckd1["event"], ckd0["event"])
R["p_age"], R["p_ckd"] = float(lr_age.p_value), float(lr_ckd.p_value)
axes[0].text(0.98, 0.98, f"log-rank {fmt_p(R['p_age'])}", transform=axes[0].transAxes,
             ha="right", va="top", fontsize=11, color="#52514e")
axes[1].text(0.98, 0.98, f"log-rank {fmt_p(R['p_ckd'])}", transform=axes[1].transAxes,
             ha="right", va="top", fontsize=11, color="#52514e")
save(fig, "fig2_km_groups")

for k, v in {**R["death_by_age"], **R["death_by_ckd"]}.items():
    print(f"{k:12s} 90 day death {v:.1f}%")
print("age groups:", fmt_p(R["p_age"]), "| CKD:", fmt_p(R["p_ckd"]))

## Step 7: train/test split and prep

Now the models. I hold out 25% of patients as a test set that the models never see during training. It's stratified on `event` so both sets have the same share of deaths.

Prep, all learned from the **training set only** and then applied to both (so nothing leaks from the test set):
1. Fill missing labs with the training median
2. Take the log of creatinine and BUN, they're very skewed (most people normal, a few very high)
3. Scale the continuous columns to mean 0 and sd 1, so the Cox hazard ratios are "per 1 standard deviation" and easy to compare

In [ ]:
train, test = train_test_split(df, test_size=0.25, random_state=SEED, stratify=df["event"])
R["n_train"], R["n_test"] = len(train), len(test)
print(f"train {len(train):,} patients | test {len(test):,} patients")

CONT = ["age", "creatinine", "bun", "sodium", "hemoglobin"]
medians = train[LABS].median()          # from train only


def fill_and_log(d):
    """Features only, missing labs filled with the train median, log of the 2 skewed labs."""
    X = d[FEATURES].copy()
    X[LABS] = X[LABS].fillna(medians)
    X[["creatinine", "bun"]] = np.log(X[["creatinine", "bun"]])
    return X


X_train, X_test = fill_and_log(train), fill_and_log(test)
means, sds = X_train[CONT].mean(), X_train[CONT].std()   # from train only
X_train[CONT] = (X_train[CONT] - means) / sds
X_test[CONT] = (X_test[CONT] - means) / sds
R["sd_age"] = float(sds["age"])

print(X_train.describe().loc[["mean", "std"]].round(2).to_string())

## Step 8: Cox proportional hazards model

KM only compares one grouping at a time. The Cox model looks at all features together.

- **Hazard** = the risk of dying at a given moment, given you're still alive
- Cox gives each feature a **hazard ratio (HR)**. HR 1.5 = 50% higher hazard, HR 0.8 = 20% lower, HR 1 = no effect
- For the scaled features the HR is per 1 standard deviation, for 0/1 features it's yes vs no
- It assumes each HR stays the same for the whole 90 days (the "proportional hazards" assumption). I check that further down.

In [ ]:
# lifelines wants one dataframe with the features plus the time and event columns
cox_train = X_train.assign(time=train["time"], event=train["event"])

cph = CoxPHFitter()
cph.fit(cox_train, duration_col="time", event_col="event")

hr = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
hr.columns = ["HR", "low", "high", "p"]
hr = hr.sort_values("HR")
print(hr.round(3).to_string())

R["cox_hr"] = {f: [float(v) for v in hr.loc[f, ["HR", "low", "high", "p"]]] for f in hr.index}
sig = hr[hr["p"] < 0.05]
R["cox_up"] = [[NAMES[f], float(h)] for f, h in sig[sig["HR"] > 1]["HR"].sort_values(ascending=False).items()]
R["cox_down"] = [[NAMES[f], float(h)] for f, h in sig[sig["HR"] < 1]["HR"].sort_values().items()]

In [ ]:
# forest plot: dot = HR, line = 95% CI, blue if p < 0.05
fig, ax = plt.subplots(figsize=(8, 5.6))
y = np.arange(len(hr))
colors = np.where(hr["p"] < 0.05, BLUE, GRAY)
ax.hlines(y, hr["low"], hr["high"], color=colors, lw=2)
ax.scatter(hr["HR"], y, color=colors, s=60, zorder=3)
ax.axvline(1, color="#52514e", lw=1, ls="--")
ax.set_xscale("log")
ticks = [t for t in [0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 6] if hr["low"].min() / 1.2 <= t <= hr["high"].max() * 1.2]
ax.set_xticks(ticks, [str(t) for t in ticks])
ax.minorticks_off()
ax.set_yticks(y, [NAMES[f] for f in hr.index])
ax.grid(axis="y", visible=False)
for yi, h in zip(y, hr["HR"]):          # HR values as a column on the right
    ax.text(1.02, yi, f"{h:.2f}", transform=ax.get_yaxis_transform(), va="center", fontsize=10, color=INK)
ax.text(1.02, len(hr) - 0.3, "HR", transform=ax.get_yaxis_transform(), fontsize=10, color="#52514e")
ax.set_xlabel("Hazard ratio (log scale), 1 = no effect")
ax.set_title("Cox model: hazard ratio for each feature", loc="left")
ax.legend(handles=[Line2D([], [], color=BLUE, marker="o", lw=2, label="p < 0.05"),
                   Line2D([], [], color=GRAY, marker="o", lw=2, label="not significant")],
          loc="lower right")
save(fig, "fig3_cox_hazard_ratios")

In [ ]:
def c_index(risk, d):
    """Harrell's C-index on dataframe d: out of all pairs of patients we can compare,
    how often the one who died first got the higher risk. 0.5 = coin flip, 1 = perfect."""
    return concordance_index_censored(d["event"].astype(bool).to_numpy(),
                                      d["time"].to_numpy(), np.asarray(risk))[0]


# risk score for Cox = partial hazard (bigger = higher risk)
risk_cox = cph.predict_partial_hazard(X_test).to_numpy()

# simple baseline to beat: a Cox model with age only
cph_age = CoxPHFitter().fit(cox_train[["age", "time", "event"]], "time", "event")
risk_age = cph_age.predict_partial_hazard(X_test[["age"]]).to_numpy()

R["c_age"], R["c_cox"] = c_index(risk_age, test), c_index(risk_cox, test)
print(f"test C-index   age only: {R['c_age']:.3f}   Cox all features: {R['c_cox']:.3f}")

In [ ]:
# proportional hazards check: a small p means that feature's effect changes over the 90 days
ph = proportional_hazard_test(cph, cox_train, time_transform="rank")
ph_p = ph.summary["p"]
R["ph_flagged"] = [NAMES[f] for f in ph_p.index if ph_p[f] < 0.05]
print(ph_p.round(3).to_string())
print("flagged (p < 0.05):", R["ph_flagged"] or "none")

## Step 9: Random Survival Forest

The machine learning model. Same idea as a normal random forest, but built for survival data:

- Each tree is trained on a random bootstrap sample of the training patients
- At each split it picks the feature and cutoff that make the two groups' survival as different as possible (a log-rank split)
- Each leaf holds a survival curve for the patients that end up in it, and the forest averages the curves from all trees
- No proportional hazards assumption, and it picks up non-linear effects and interactions on its own

scikit-survival wants the outcome as one special array with both the event (True/False) and the time, `Surv.from_arrays` makes that.

In [ ]:
y_train = Surv.from_arrays(event=train["event"].astype(bool), time=train["time"])
y_test = Surv.from_arrays(event=test["event"].astype(bool), time=test["time"])

rsf = RandomSurvivalForest(
    n_estimators=300,        # number of trees, more = more stable but slower
    min_samples_leaf=15,     # each leaf needs at least 15 patients, stops trees memorizing single people
    max_features="sqrt",     # each split only looks at a random sqrt(12) ~ 3 features, makes trees different
    n_jobs=-1,               # use all cpu cores
    random_state=SEED,
)
rsf.fit(X_train, y_train)

risk_rsf = rsf.predict(X_test)      # risk score, bigger = higher risk
R["c_rsf"] = c_index(risk_rsf, test)
print(f"test C-index   Random Survival Forest: {R['c_rsf']:.3f}")

## Step 10: compare the models with the C-index

The **C-index** (concordance index) is the usual score for survival models. Take every pair of test patients where we know who died first. The C-index is the share of those pairs where the model gave the higher risk to the person who died first. 0.5 = coin flip, 1.0 = perfect. Unlike accuracy it works with censored people.

One number can be lucky, so I **bootstrap** the test set 200 times (resample patients with replacement, recompute) to get a 95% interval for each model, and for the gap between RSF and Cox.

In [ ]:
risks = {"Age only (Cox)": risk_age, "Cox, all features": risk_cox, "Random Survival Forest": risk_rsf}
rng = np.random.default_rng(SEED)
boot = {k: [] for k in risks}
for _ in range(200):
    idx = rng.integers(0, len(test), len(test))      # resample test patients with replacement
    d = test.iloc[idx]
    for k, r in risks.items():
        boot[k].append(c_index(r[idx], d))
boot = {k: np.array(v) for k, v in boot.items()}

R["c_ci"] = {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in boot.items()}
diff = boot["Random Survival Forest"] - boot["Cox, all features"]
R["diff_ci"] = [float(np.percentile(diff, 2.5)), float(np.percentile(diff, 97.5))]
R["rsf_wins_pct"] = float(100 * (diff > 0).mean())
for k in risks:
    lo, hi = R["c_ci"][k]
    print(f"{k:24s} {c_index(risks[k], test):.3f}  (95% CI {lo:.3f} to {hi:.3f})")
print(f"RSF minus Cox: 95% CI {R['diff_ci'][0]:+.3f} to {R['diff_ci'][1]:+.3f}, "
      f"RSF higher in {R['rsf_wins_pct']:.0f}% of bootstraps")

# dot plot (not bars, since the axis doesnt start at 0)
fig, ax = plt.subplots(figsize=(8, 3.8))
names = list(risks)
vals = [c_index(risks[k], test) for k in names]
for i, (k, v) in enumerate(zip(names, vals)):
    lo, hi = R["c_ci"][k]
    ax.hlines(i, lo, hi, color=BLUE, lw=2)
    ax.scatter(v, i, color=BLUE, s=70, zorder=3)
    ax.annotate(f"{v:.3f}", (hi, i), xytext=(8, 0), textcoords="offset points", va="center", fontsize=11)
ax.axvline(0.5, color="#52514e", ls="--", lw=1)
ax.text(0.505, len(names) - 0.6, "0.5 = coin flip", fontsize=10, color="#52514e")
ax.set_yticks(range(len(names)), names)
ax.set_ylim(-0.6, len(names) - 0.3)
ax.set_xlim(0.48, max(R["c_ci"][k][1] for k in names) + 0.05)
ax.grid(axis="y", visible=False)
ax.set_xlabel("C-index on the test set (dot) with bootstrap 95% CI (line)")
ax.set_title("Which model ranks patients by risk best?", loc="left")
save(fig, "fig4_c_index")

## Step 11: which features matter to the forest?

Random forests dont give hazard ratios. **Permutation importance** shuffles one feature at a time in the test set and checks how much the C-index drops. A big drop = the model really relies on that feature. Near 0 = the model could do without it. (This cell takes a minute or two.)

In [ ]:
# rsf.score() is the C-index, so this measures the drop in C-index when each feature is shuffled
perm = permutation_importance(rsf, X_test, y_test, n_repeats=5, random_state=SEED, n_jobs=-1)
imp = pd.DataFrame({"mean": perm.importances_mean, "sd": perm.importances_std},
                   index=X_test.columns).sort_values("mean")
print(imp.sort_values("mean", ascending=False).round(4).to_string())

R["perm"] = {NAMES[f]: float(v) for f, v in imp["mean"].sort_values(ascending=False).items()}
R["perm_top"] = list(R["perm"])[:3]

fig, ax = plt.subplots(figsize=(8, 5.6))
ax.barh([NAMES[f] for f in imp.index], imp["mean"], xerr=imp["sd"], color=BLUE, height=0.6,
        error_kw={"ecolor": "#52514e", "lw": 1})
ax.axvline(0, color="#52514e", lw=1)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Drop in C-index when the feature is shuffled")
ax.set_title("Random Survival Forest: permutation importance", loc="left")
save(fig, "fig5_rsf_importance")

## Step 12: does it separate low and high risk patients?

The C-index is just one number, so here's a more visual check. I split the test patients into 3 equal groups by the forest's predicted risk, then draw a normal KM curve for each group using what really happened to them. If the model is useful the curves should be far apart.

On the right is what the forest predicts for two single test patients, one low risk and one high risk. That's something Cox doesnt give you as directly: a full survival curve per person.

In [ ]:
test_r = test.assign(risk=risk_rsf)
test_r["risk_group"] = pd.qcut(test_r["risk"], 3, labels=["Low", "Medium", "High"]).astype(str)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
R["death_by_risk"] = {}
for g, color in zip(["Low", "Medium", "High"], RAMP):
    d = test_r[test_r["risk_group"] == g]
    k = KaplanMeierFitter().fit(d["time"], d["event"], label=f"{g} predicted risk (n={len(d):,})")
    k.plot_survival_function(ax=axes[0], color=color, lw=2, ci_alpha=0.15)
    s90 = float(k.survival_function_at_times(HORIZON).iloc[0])
    axes[0].annotate(f"{100 * s90:.0f}%", (HORIZON, s90), xytext=(5, 0),
                     textcoords="offset points", va="center", fontsize=11, color=INK)
    R["death_by_risk"][g] = 100 * (1 - s90)
R["p_risk"] = float(multivariate_logrank_test(test_r["time"], test_r["risk_group"], test_r["event"]).p_value)
axes[0].set_title("Real survival of test patients, by predicted risk", loc="left")
axes[0].set_ylabel("Probability still alive")
axes[0].legend(loc="lower left")

# two example patients: the 10th and 90th percentile of predicted risk
order = np.argsort(risk_rsf)
examples = {"Low risk patient": order[int(0.1 * len(order))], "High risk patient": order[int(0.9 * len(order))]}
R["example_s90"] = {}
for (name, i), color in zip(examples.items(), [BLUE, ORANGE]):
    fn = rsf.predict_survival_function(X_test.iloc[[i]])[0]
    axes[1].step(np.r_[0, fn.x], np.r_[1, fn.y], where="post", color=color, lw=2, label=name)
    s90 = float(fn(HORIZON))
    axes[1].annotate(f"{100 * s90:.0f}%", (HORIZON, s90), xytext=(5, 0),
                     textcoords="offset points", va="center", fontsize=11, color=INK)
    R["example_s90"][name] = 100 * s90
axes[1].set_title("Forest's predicted curve for 2 test patients", loc="left")
axes[1].legend(loc="lower left")

lowest = 1 - max(R["death_by_risk"].values()) / 100
for ax in axes:
    ax.set_xlim(0, HORIZON)
    ax.set_xticks(range(0, HORIZON + 1, 15))
    ax.set_ylim(y_floor(min(lowest, min(R["example_s90"].values()) / 100)), 1.0)
    ax.set_xlabel("Days since admission")
save(fig, "fig6_risk_groups")

for g, v in R["death_by_risk"].items():
    print(f"{g:7s} predicted risk: {v:.1f}% died within 90 days")
print("log-rank across the 3 groups:", fmt_p(R["p_risk"]))

## Summary

The numbers below come from this run. They're also saved to `results.json`.

In [ ]:
diff_lo, diff_hi = R["diff_ci"]
if diff_lo > 0:
    R["verdict"] = "The Random Survival Forest ranked patients better than the Cox model"
elif diff_hi < 0:
    R["verdict"] = "The Cox model ranked patients better than the Random Survival Forest"
else:
    R["verdict"] = "The Random Survival Forest and the Cox model did about the same (the gap could be chance)"

lines = [
    f"Cohort: {R['n']:,} adults with a first heart failure admission, {R['death_pct']:.1f}% died within 90 days",
    f"Kaplan-Meier: {R['km_30']:.1f}% alive at 30 days, {R['km_90']:.1f}% at 90 days",
    f"Age groups 90 day death: " + ", ".join(f"{k} {v:.0f}%" for k, v in R["death_by_age"].items())
    + f" ({fmt_p(R['p_age'])})",
    f"Cox, biggest risk factors: " + (", ".join(f"{n} (HR {h:.2f})" for n, h in R["cox_up"][:3]) or "none significant"),
    f"C-index on test: age only {R['c_age']:.3f}, Cox {R['c_cox']:.3f}, RSF {R['c_rsf']:.3f}",
    R["verdict"],
    f"Most important features for the forest: {', '.join(R['perm_top'])}",
    f"Forest risk groups, real 90 day death: " + ", ".join(f"{k} {v:.0f}%" for k, v in R["death_by_risk"].items()),
]
print("\n".join(lines))

with open("results.json", "w") as f:
    json.dump(R, f, indent=2)

### Limitations and what I'd try next

- One hospital (Beth Israel Deaconess in Boston) and only patients who went to the ICU, so it might not carry over to other places
- Diagnosis codes are added at discharge, so the comorbidities arent 100% "known at admission"
- Only 90 days of follow up, and no ejection fraction, BNP or medications, which matter alot in heart failure
- Next: add vitals and meds, try gradient boosted survival models or DeepSurv (a neural network Cox model), check calibration, test it on MIMIC-IV

### References
- Johnson AEW et al. MIMIC-III, a freely accessible critical care database. *Scientific Data* 3, 160035 (2016)
- Kaplan EL, Meier P. Nonparametric estimation from incomplete observations. *JASA* 53 (1958)
- Cox DR. Regression models and life-tables. *J R Stat Soc B* 34 (1972)
- Ishwaran H, Kogalur UB, Blackstone EH, Lauer MS. Random survival forests. *Annals of Applied Statistics* 2(3) (2008)
- Harrell FE et al. Evaluating the yield of medical tests. *JAMA* 247 (1982)
- lifelines docs: https://lifelines.readthedocs.io | scikit-survival docs: https://scikit-survival.readthedocs.io

In [ ]:
# download the figures (and the slides, if they were built) to your computer
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("figures", "zip", ".", "figures")
    for f in ["figures.zip", "results.json", "HF_Survival_Tutorial.pptx"]:
        if os.path.exists(f):
            try:
                files.download(f)
            except Exception as e:
                print(f"couldnt download {f}, grab it from the file browser on the left ({e})")